# SpaceX Falcon 9 — Interactive Map with Folium

**Author:** Piyu

We build an interactive map of SpaceX launch sites, marking each site's launches as a cluster of success/failure markers, and compute the distance from a launch site to nearby geographical features (coastline, city, railway).

In [1]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster
from math import sin, cos, sqrt, atan2, radians

df = pd.read_csv('../data/dataset_part_2.csv')
sites = df[['LaunchSite','Longitude','Latitude']].drop_duplicates().reset_index(drop=True)
sites

,LaunchSite,Longitude,Latitude
0,CCAFS SLC 40,-80.577366,28.561857
1,VAFB SLC 4E,-120.610829,34.632093
2,KSC LC 39A,-80.603956,28.608058


## 1. Mark all launch sites on a global map

In [2]:
site_map = folium.Map(location=[28.5, -80.6], zoom_start=4)

for _, row in sites.iterrows():
    folium.Circle(
        [row['Latitude'], row['Longitude']],
        radius=1000, color='#0066cc', fill=True
    ).add_child(folium.Popup(row['LaunchSite'])).add_to(site_map)
    folium.map.Marker(
        [row['Latitude'], row['Longitude']],
        icon=folium.DivIcon(icon_size=(20,20), icon_anchor=(0,0),
            html=f'<div style="font-size:12px;color:#0066cc;"><b>{row["LaunchSite"]}</b></div>')
    ).add_to(site_map)

site_map.save('../images/launch_sites_map.html')
site_map

## 2. Color-labeled markers for success/failure, clustered by site

In [3]:
marker_cluster = MarkerCluster().add_to(site_map)

df['marker_color'] = df['Class'].apply(lambda x: 'green' if x == 1 else 'red')

for _, row in df.iterrows():
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        icon=folium.Icon(color='white', icon_color=row['marker_color']),
        popup=f"{row['LaunchSite']} - {'Success' if row['Class']==1 else 'Failure'}"
    ).add_to(marker_cluster)

site_map.save('../images/launch_sites_clustered.html')
site_map

## 3. Success rate per launch site (from marker data)

Using the color-labeled clusters, we can directly see which sites have a higher concentration of green (successful) markers.

In [4]:
site_success = df.groupby('LaunchSite')['Class'].agg(['mean','count']).rename(columns={'mean':'SuccessRate','count':'Launches'})
site_success.sort_values('SuccessRate', ascending=False)

,SuccessRate,Launches
LaunchSite,,
KSC LC 39A,0.772727,22
VAFB SLC 4E,0.769231,13
CCAFS SLC 40,0.600000,55


## 4. Distance from a launch site to nearby landmarks

In [5]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R * c

# KSC LC-39A coordinates, and an approximate nearby coastline point
launch_site_lat, launch_site_lon = 28.6080, -80.6040
coastline_lat, coastline_lon = 28.5631, -80.5678  # approx nearest coastline point

dist_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
print(f"Distance from KSC LC-39A to nearest coastline: {dist_coastline:.2f} km")

Distance from KSC LC-39A to nearest coastline: 6.12 km


In [6]:
# Draw a line from the launch site to the coastline point on the map
distance_map = folium.Map(location=[launch_site_lat, launch_site_lon], zoom_start=12)

folium.Marker([launch_site_lat, launch_site_lon], popup='KSC LC-39A', icon=folium.Icon(color='blue')).add_to(distance_map)
folium.Marker([coastline_lat, coastline_lon], popup='Nearest coastline', icon=folium.Icon(color='gray')).add_to(distance_map)
folium.PolyLine(
    locations=[[launch_site_lat, launch_site_lon],[coastline_lat, coastline_lon]],
    weight=2, color='#444'
).add_to(distance_map)
folium.map.Marker(
    [(launch_site_lat+coastline_lat)/2, (launch_site_lon+coastline_lon)/2],
    icon=folium.DivIcon(html=f'<div style="font-size:12px;">{dist_coastline:.2f} KM</div>')
).add_to(distance_map)

distance_map.save('../images/distance_map.html')
distance_map

## Summary

- All 3 SpaceX launch sites in the dataset are plotted on a global map, all located close to the equator and close to the coast
- Color-labeled clustered markers make it visually clear which sites have a higher proportion of successful landings
- Launch sites are positioned near coastlines (for safety — so failed boosters fall into the ocean rather than over land), confirmed by the calculated distance to the nearest coastline point